# 02 - Architecture Tour

Welcome to the model. This notebook is a hands-on tour of the deep residual 1D CNN that classifies gravitational wave signals in the G2Net dataset. We walk through every stage of the network, poke at the shapes, count the parameters, and get a feel for why the thing is put together the way it is.

For the data side of the story, start with [01_data_exploration.ipynb](01_data_exploration.ipynb). For how the sky-geometry features enter the picture, see [03_sky_features.ipynb](03_sky_features.ipynb). For performance numbers and error analysis, see [04_results_and_errors.ipynb](04_results_and_errors.ipynb).

> **No trained weights needed here.** We instantiate a fresh model, push a fake batch through, and let the shapes speak.


## 1. Setup

In [1]:
from __future__ import annotations
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from models.diy_model import DIYModel, GeM, ResBlock, SkyFiLM, AdaptiveConcatPool1d

torch.manual_seed(0)
print(f"torch: {torch.__version__}")


torch: 2.9.1+cpu


## 2. Instantiate the Model

The model has three knobs worth knowing about:

- `n_channels` - base channel width. Channels widen through the backbone as `n -> n -> 2n -> 4n`. Default `16`, which keeps the Kaggle run under the 9-hour P100 budget.
- `dropout_rate` - dropout in the classifier head. Set to `0.5`.
- `drop_path_rate` - maximum stochastic depth. Linearly scales from `0` at the first block to `drop_path_rate` at the last.
- `n_sky_features` - number of spherical harmonic coefficients fed into FiLM. With `l_max=10`, that's `121`. More on this in notebook 03.


In [2]:
model = DIYModel(
    n_channels=16,
    dropout_rate=0.5,
    drop_path_rate=0.3,
    n_sky_features=121,
)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {n_params:>12,}")
print(f"Trainable parameters: {n_trainable:>12,}")


Total parameters:        4,703,964
Trainable parameters:    4,703,964


A little under 5 million parameters. The fusion stage dominates, as we'll see in section 5. Deep enough that a plain stack of convolutions would choke on vanishing gradients long before it learned anything useful. That's what the residual connections are for.

## 3. A Fake Batch

We'll push a synthetic batch through the network and print the shape after every major stage. Real data comes in as `(B, 3, 4096)` - batch, detectors (H1, L1, V1), and 2 seconds at 2048 Hz.


In [3]:
B = 4  # batch size
X = torch.randn(B, 3, 4096)
sky = torch.randn(B, 121)  # SH coefficients

print(f"Input X:   {tuple(X.shape)}")
print(f"Input sky: {tuple(sky.shape)}")


Input X:   (4, 3, 4096)
Input sky: (4, 121)


## 4. Stage-by-Stage Walkthrough

The forward pass has five stages. We'll break them apart and watch the tensor shapes evolve. This is also where the V2 two-stage fusion topology becomes visible: individual branches for each detector plus a joint branch, merged and refined.


### 4.1 Extractors

Each detector goes through its own extractor: a pair of large kernel convolutions (`k=64`, about 31 ms of signal) followed by a GeM pool that halves the temporal resolution. H1 and L1 share weights (same instrument type, same sensitivity). Virgo gets its own because it's a different instrument.


In [4]:
with torch.no_grad():
    h1 = model.ligo_extractor(X[:, 0:1, :])
    l1 = model.ligo_extractor(X[:, 1:2, :])
    v1 = model.virgo_extractor(X[:, 2:3, :])

print(f"After extractor -- H1: {tuple(h1.shape)}")
print(f"After extractor -- L1: {tuple(l1.shape)}")
print(f"After extractor -- V1: {tuple(v1.shape)}")
print(f"\nTemporal dim: 4096 -> {h1.shape[-1]} (halved by GeM)")
print(f"Channels:     1 -> {h1.shape[1]}")


After extractor -- H1: (4, 16, 2048)
After extractor -- L1: (4, 16, 2048)
After extractor -- V1: (4, 16, 2048)

Temporal dim: 4096 -> 2048 (halved by GeM)
Channels:     1 -> 16


c:\Users\jose\OneDrive\Ambiente de Trabalho\lisa_gravitational_wave_detector\.venv\Lib\site-packages\torch\nn\modules\conv.py:366: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1032.)
  return F.conv1d(


### 4.2 Residual Backbone

Ten residual blocks organized into five groups. Each block is two `Conv1d -> BN -> SiLU` pairs with a shortcut that skips around them. The downsampling groups (marked with `GeM(k=4)`) cut temporal resolution by 4x each time; identity groups just refine features at the current resolution.

| Group | Channels | Kernel | Downsample | Temporal |
|-------|----------|--------|------------|----------|
| 1     | n -> n   | 31     | 4          | 2048 -> 512 |
| 2     | n        | 31     | -          | 512 |
| 3     | n -> 2n  | 15     | 4          | 512 -> 128 |
| 4     | 2n -> 4n | 7      | 4          | 128 -> 32 |
| 5     | 4n       | 7      | -          | 32 |

The three detectors are concatenated along the batch dimension so the backbone sees `(3B, n, 2048)` in one shot. Weight sharing comes for free because they're all flowing through the same Sequential.


In [5]:
with torch.no_grad():
    stacked_in = torch.cat([h1, l1, v1], dim=0)
    print(f"Backbone input:  {tuple(stacked_in.shape)}  (3 * {B} detectors batched)")
    stacked_out = model.backbone(stacked_in)
    print(f"Backbone output: {tuple(stacked_out.shape)}")

h1_feat, l1_feat, v1_feat = stacked_out.chunk(3, dim=0)
print(f"\nPer-detector features: {tuple(h1_feat.shape)}")
print(f"Temporal compression: 2048 -> {h1_feat.shape[-1]} (64x from 4096)")
print(f"Channel widening:     16 -> {h1_feat.shape[1]} (4x)")


Backbone input:  (12, 16, 2048)  (3 * 4 detectors batched)
Backbone output: (12, 64, 32)

Per-detector features: (4, 64, 32)
Temporal compression: 2048 -> 32 (64x from 4096)
Channel widening:     16 -> 64 (4x)


### 4.3 V2 Branch Paths (Stage 1 of Fusion)

Here's where the V2-style two-stage fusion begins. After the shared backbone, the features split into **four parallel paths**:

1. **H1 branch** - LIGO branch blocks, processing H1 features alone
2. **L1 branch** - same LIGO branch blocks (weights shared with H1), processing L1
3. **V1 branch** - Virgo branch blocks (separate weights), processing V1
4. **Joint branch** - all three detectors concatenated channel-wise, projected back down, then processed through joint branch blocks

The joint branch is the interesting one: it lets the network look at all three detectors at the same time while the individual branches refine per-detector features in parallel. Two views of the same evidence.


In [6]:
with torch.no_grad():
    ligo_stacked_in = torch.cat([h1_feat, l1_feat], dim=0)
    ligo_stacked_out = model.ligo_branch(ligo_stacked_in)
    h1_branch, l1_branch = ligo_stacked_out.chunk(2, dim=0)
    v1_branch = model.virgo_branch(v1_feat)

    joint_in = torch.cat([h1_feat, l1_feat, v1_feat], dim=1)
    joint_projected = model.joint_project(joint_in)
    joint_branch = model.joint_branch(joint_projected)

print(f"H1 branch:    {tuple(h1_branch.shape)}")
print(f"L1 branch:    {tuple(l1_branch.shape)}")
print(f"V1 branch:    {tuple(v1_branch.shape)}")
print(f"\nJoint branch input (concat): {tuple(joint_in.shape)}  ({3 * h1_feat.shape[1]} channels)")
print(f"Joint branch after 1x1 proj: {tuple(joint_projected.shape)}")
print(f"Joint branch output:         {tuple(joint_branch.shape)}")


H1 branch:    (4, 64, 32)
L1 branch:    (4, 64, 32)
V1 branch:    (4, 64, 32)

Joint branch input (concat): (4, 192, 32)  (192 channels)
Joint branch after 1x1 proj: (4, 64, 32)
Joint branch output:         (4, 64, 32)


### 4.4 Fusion Blocks (Stage 2 of Fusion)

Four branch paths concatenated along the channel axis, then four more residual blocks work on the merged tensor. This is where the network gets to mix individual-detector features with the joint view. It's also the widest point in the architecture: `4 * 4n = 16n` channels.


In [7]:
with torch.no_grad():
    fused_in = torch.cat([h1_branch, l1_branch, v1_branch, joint_branch], dim=1)
    print(f"Merged 4 branches: {tuple(fused_in.shape)}  ({fused_in.shape[1]} channels)")
    fused = model.fusion(fused_in)
print(f"After fusion blocks: {tuple(fused.shape)}")


Merged 4 branches: (4, 256, 32)  (256 channels)
After fusion blocks: (4, 256, 32)


### 4.5 Global Pool, Sky FiLM, Classifier

AdaptiveConcatPool1d collapses the temporal dimension by computing both adaptive avg-pool and adaptive max-pool, then concatenating them - so the channel count doubles one last time to `32n = 512`.

Then the sky FiLM module takes over. Sky features (more on those in notebook 03) feed into a small MLP that produces a per-channel `(gamma, beta)`. The pooled features become `(1 + gamma) * features + beta`, with both `gamma` and `beta` tanh-bounded so the modulation can't run away. The output projection is zero-initialized so at step 0 the FiLM layer is exactly the identity - the optimizer has to *earn* any use of the sky path.

Finally, a three-layer classifier head spits out raw logits (no sigmoid; we use `BCEWithLogitsLoss`).


In [8]:
with torch.no_grad():
    pooled = model.global_pool(fused).squeeze(-1)
    print(f"After pool (avg+max concat):   {tuple(pooled.shape)}")
    modulated = model.sky_film(pooled, sky)
    print(f"After SkyFiLM modulation:      {tuple(modulated.shape)}")
    logits = model.classifier(modulated)
    print(f"After classifier head (logit): {tuple(logits.shape)}")
    probs = torch.sigmoid(logits).squeeze().numpy()
    print(f"\nProbabilities (random init, so ~0.5): {np.round(probs, 3)}")


After pool (avg+max concat):   (4, 512)
After SkyFiLM modulation:      (4, 512)
After classifier head (logit): (4, 1)

Probabilities (random init, so ~0.5): [0. 0. 0. 0.]


### 4.6 Sanity Check: One-Shot Forward

The five stages above are exactly what the model does in `forward()`. Let's confirm by calling it directly and comparing logits.


In [9]:
with torch.no_grad():
    logits_full = model(X, sky_features=sky)

print(f"Full forward logits: {tuple(logits_full.shape)}")
print(f"Match stage-by-stage? {torch.allclose(logits_full, logits)}")


Full forward logits: (4, 1)
Match stage-by-stage? True


## 5. Parameter Budget by Section

Where does that ~850k parameter count actually go? Let's see.


In [10]:
def count_params(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters())


sections = {
    "LIGO extractor (shared H1/L1)": model.ligo_extractor,
    "Virgo extractor":                model.virgo_extractor,
    "Residual backbone (10 blocks)": model.backbone,
    "LIGO branch (2 blocks, shared)": model.ligo_branch,
    "Virgo branch (2 blocks)":        model.virgo_branch,
    "Joint projection (1x1 conv)":    model.joint_project,
    "Joint branch (2 blocks)":        model.joint_branch,
    "Fusion (4 blocks)":              model.fusion,
    "Sky FiLM":                       model.sky_film,
    "Classifier head":                model.classifier,
    "Branch aux head":                model.branch_head,
}

total = 0
for name, mod in sections.items():
    c = count_params(mod)
    total += c
    print(f"  {name:<35s} {c:>10,}")
print(f"  {'-' * 50}")
print(f"  {'Total':<35s} {total:>10,}")


  LIGO extractor (shared H1/L1)           17,473
  Virgo extractor                         17,473
  Residual backbone (10 blocks)          337,446
  LIGO branch (2 blocks, shared)         115,456
  Virgo branch (2 blocks)                115,456
  Joint projection (1x1 conv)             12,480
  Joint branch (2 blocks)                115,456
  Fusion (4 blocks)                    3,676,160
  Sky FiLM                               147,954
  Classifier head                        148,481
  Branch aux head                            129
  --------------------------------------------------
  Total                                4,703,964


The fusion blocks dominate - roughly 3.7M of the 4.7M parameters live there, since they operate at `16n = 256` channels with kernel 7. That's where most of the heavy lifting happens after the branch stage. The sky FiLM module is almost free: a BatchNorm, two linear layers. And yet it carries the geometric prior that took the whole ablation phase to wire in properly.

## 6. The GeM Pooling Quirk

GeM (Generalized Mean pooling) interpolates between average and max pooling via a **learnable** exponent `p`. Average pooling is `p=1`, max pooling is the limit as `p -> infty`. The network picks what it wants.

The formula: `(mean(x^p))^(1/p)`. Because it takes `x^p`, negative activations blow up unless you clamp them first - hence the `clamp(min=1e-6)`. And because the exponent can drift (AdamW weight decay tries to pull every parameter toward zero, including `p`), the forward pass clamps `p` to `[1, 10]` as belt-and-suspenders.

Let's see what the freshly-initialized `p` values are across the network:


In [11]:
gem_params = [(name, m.p.item()) for name, m in model.named_modules() if isinstance(m, GeM)]
print(f"Number of GeM modules: {len(gem_params)}\n")
for name, p in gem_params:
    print(f"  p = {p:.2f}   {name}")


Number of GeM modules: 8

  p = 3.00   ligo_extractor.4
  p = 3.00   virgo_extractor.4
  p = 3.00   backbone.0.pool
  p = 3.00   backbone.0.shortcut.0
  p = 3.00   backbone.4.pool
  p = 3.00   backbone.4.shortcut.2
  p = 3.00   backbone.6.pool
  p = 3.00   backbone.6.shortcut.2


All initialized to `p = 3`, somewhere between mean and max. If training goes well, some of these will drift up or down depending on which strategy works for that layer. A trained model usually shows a spread.

## 7. Receptive Field

One of the key reasons for going deep: the receptive field needs to cover the whole 2-second window. Our old 4-block network only saw ~28% of the input; the signal's amplitude envelope was wider than what any individual output unit could look at.

The deep architecture fixes this. Here's a theoretical receptive field calculation, walking through each conv op:


In [12]:
def receptive_field_trace(ops: list[tuple[str, int, int]]) -> None:
    '''ops is a list of (name, kernel_size, stride_after_op).'''
    rf = 1
    stride = 1
    print(f"  {'operation':<30s} {'k':>4s} {'rf':>6s} {'stride':>8s}")
    print(f"  {'-' * 52}")
    for name, k, s in ops:
        rf = rf + (k - 1) * stride
        stride = stride * s
        print(f"  {name:<30s} {k:>4d} {rf:>6d} {stride:>8d}")
    print(f"\n  Final receptive field: {rf} samples ({rf / 2048 * 1000:.0f} ms at 2048 Hz)")
    print(f"  Input length: 4096 samples (2000 ms)")
    coverage = min(rf / 4096, 1.0) * 100
    print(f"  Coverage: {coverage:.0f}%")


# Approximate trace of the main path (ignoring shortcut convs)
ops = [
    ("extractor conv1 k=64",    64, 1),
    ("extractor conv2 k=64",    64, 1),
    ("extractor GeM(2)",         2, 2),
    ("group1 block1 conv1 k=31", 31, 1),
    ("group1 block1 conv2 k=31", 31, 1),
    ("group1 block1 GeM(4)",      4, 4),
    ("group1 block2 conv1 k=31", 31, 1),
    ("group1 block2 conv2 k=31", 31, 1),
    ("group2 block1 conv1 k=31", 31, 1),
    ("group2 block1 conv2 k=31", 31, 1),
    ("group2 block2 conv1 k=31", 31, 1),
    ("group2 block2 conv2 k=31", 31, 1),
    ("group3 block1 conv1 k=15", 15, 1),
    ("group3 block1 conv2 k=15", 15, 1),
    ("group3 block1 GeM(4)",      4, 4),
    ("group3 block2 conv1 k=15", 15, 1),
    ("group3 block2 conv2 k=15", 15, 1),
    ("group4 block1 conv1 k=7",   7, 1),
    ("group4 block1 conv2 k=7",   7, 1),
    ("group4 block1 GeM(4)",      4, 4),
    ("group4 block2 conv1 k=7",   7, 1),
    ("group4 block2 conv2 k=7",   7, 1),
]

receptive_field_trace(ops)


  operation                         k     rf   stride
  ----------------------------------------------------
  extractor conv1 k=64             64     64        1
  extractor conv2 k=64             64    127        1
  extractor GeM(2)                  2    128        2
  group1 block1 conv1 k=31         31    188        2
  group1 block1 conv2 k=31         31    248        2
  group1 block1 GeM(4)              4    254        8
  group1 block2 conv1 k=31         31    494        8
  group1 block2 conv2 k=31         31    734        8
  group2 block1 conv1 k=31         31    974        8
  group2 block1 conv2 k=31         31   1214        8
  group2 block2 conv1 k=31         31   1454        8
  group2 block2 conv2 k=31         31   1694        8
  group3 block1 conv1 k=15         15   1806        8
  group3 block1 conv2 k=15         15   1918        8
  group3 block1 GeM(4)              4   1942       32
  group3 block2 conv1 k=15         15   2390       32
  group3 block2 conv2 k=15 

Full coverage, with headroom. The rest of the network (group 5, branches, fusion) just refines features that already see the entire input.

Note this is *theoretical* RF. Effective RF - the region that actually contributes gradient - is smaller. But residual connections keep the effective RF closer to the theoretical bound than a plain CNN, where effective RF grows only as `O(sqrt(depth))` due to gradient attenuation. That's the other half of why residual connections matter here.


## 8. Play Time: Swap the Width

Change `n_channels` and see how the parameter count scales. Quadratic in `n` because conv weights are `O(in * out)` and both grow linearly.


In [13]:
for n in [8, 16, 24, 32]:
    m = DIYModel(n_channels=n, n_sky_features=121)
    p = sum(param.numel() for param in m.parameters())
    print(f"  n_channels={n:>3d}   params: {p:>12,}")


  n_channels=  8   params:    1,270,124
  n_channels= 16   params:    4,703,964
  n_channels= 24   params:   10,334,796
  n_channels= 32   params:   18,162,620


For the Kaggle run we stuck with `n=16`. The jump from 16 to 32 quadruples the parameter count and would put us over the 9-hour training budget on a P100. With AMP it's tight but feasible - try `n=24` next time if the fusion architecture stays this wide.

## Up Next

Now you know the shape of the network. Head to [03_sky_features.ipynb](03_sky_features.ipynb) to see where those 121 sky features actually come from, and why they're hiding in a tanh-bounded FiLM layer instead of being concatenated onto the CNN features.
